In [ ]:
import os
import xarray as xr
import datetime as dt
import matplotlib.dates as mdates
import numpy as np
import matplotlib.pyplot as plt
import gsw
import cmocean.cm as cmo
import pandas as pd
from numpy.matlib import repmat
myFmtlong = mdates.DateFormatter('%m/%d\n%H:%M')

## Set plotting parameters
SMALL_SIZE = 12
MEDIUM_SIZE = 15
BIGGER_SIZE = 20

# increase text sizes because the figure is so big
fac =1.5
plt.rc('font', size=SMALL_SIZE * fac)          # controls default text sizes
plt.rc('axes', titlesize=MEDIUM_SIZE * fac)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE * fac)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE * fac)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE * fac)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE * fac)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE * fac *0.8)  # fontsize of the figure title

In [ ]:
## Paths -- edit these to match your local setup
DATA_DIR = './data'
FORCING_DIR = f'{DATA_DIR}/forcing'
OUTPUT_DIR = f'{DATA_DIR}/final_run/data'
FIGURES_DIR = f'{DATA_DIR}/final_run/figures'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

## Load the PWP model output (barrier layer vs. no barrier layer)

In [ ]:
save_prof = 1629339767 #8/19
# save_prof =  1630114503 #8/28
profileID = save_prof

if save_prof == 1629339767:
    tstart_dt = dt.datetime(2021,8,19)
    tstart = "2021-8-19 00:00:00" # Start from near our initial condition time
    ep_idx = 996 #8/29 9z
    lf_idx = 1028 #8/29 17z
    tfreq = '2D'
elif save_prof == 1630114503:
    tstart_dt = dt.datetime(2021,8,28)
    tstart = "2021-8-28 00:00:00" # Start from near our initial condition time
    ep_idx = 132 #8/29 9z
    lf_idx = 164 #8/29 17z
    tfreq = '12H'

# Load in the PWP output for the original (barrier layer) and altered (no barrier layer) cases
force = 'HRRR'
mld_test = 'MLD1e-4'
case = str(profileID)+'_'+force+'_'+mld_test+'_'+tstart_dt.strftime('%m_%d')

pwp_orig = xr.open_dataset(f'{OUTPUT_DIR}/original_'+case+"_pwp_1m_dz_winds_drag.nc")
pwp_altr = xr.open_dataset(f'{OUTPUT_DIR}/altered_'+case+"_pwp_1m_dz_winds_drag.nc")

# The PWP output doesn't carry lat/lon, but the initial-condition file does -- load it for the gsw conversion below
_ic = xr.open_dataset(f'{OUTPUT_DIR}/ng645-'+str(profileID)+'_original.nc')
PROFILE_LAT = float(_ic['lat'][0].values)
PROFILE_LON = float(_ic['lon'][0].values)

plot_tend = "2021-8-31 00:00:00"

## Compute mixed layer depth (MLD) and isothermal layer depth (ILD) evolution for both cases

In [ ]:
bl_ild=[]
bl_mld = []
nobl_ild=[]
nobl_mld=[]

for tidx in pwp_orig.tnum.values:
    ## Find initial MLD based on Rudzin 2018 Eq1
    thresh = 0.5

    # Calculate the rho profile at tidx (pressure approximated by depth in dbar, at a fixed lat/lon)
    SA = gsw.SA_from_SP(pwp_orig['sal'][:,tidx], pwp_orig['z_vector'], PROFILE_LON, PROFILE_LAT)
    CT = gsw.CT_from_t(SA, pwp_orig['temp'][:,tidx], pwp_orig['z_vector'])
    ini_rho = gsw.rho(SA, CT, pwp_orig['z_vector'])

    aSA = gsw.SA_from_SP(pwp_altr['sal'][:,tidx], pwp_altr['z_vector'], PROFILE_LON, PROFILE_LAT)
    aCT = gsw.CT_from_t(aSA, pwp_altr['temp'][:,tidx], pwp_altr['z_vector'])
    aini_rho = gsw.rho(aSA, aCT, pwp_altr['z_vector'])
    # Calculate density at 2m depth
    SA_2m = gsw.SA_from_SP(pwp_orig['sal'][2,tidx], 2, PROFILE_LON, PROFILE_LAT)
    CT_2m = gsw.CT_from_t(SA_2m, pwp_orig['temp'][2,tidx], 2)
    rho_2m = gsw.rho(SA_2m, CT_2m, 2)

    aSA_2m = gsw.SA_from_SP(pwp_altr['sal'][2,tidx], 2, PROFILE_LON, PROFILE_LAT)
    aCT_2m = gsw.CT_from_t(aSA_2m, pwp_altr['temp'][2,tidx], 2)
    arho_2m = gsw.rho(aSA_2m, aCT_2m, 2)
    # Calculate the density if the temp was 0.5°C cooler
    CT_mld = gsw.CT_from_t(SA_2m, pwp_orig['temp'][2,tidx] - thresh, 2)
    rho_mld = gsw.rho(SA_2m, CT_mld, 2)

    aCT_mld = gsw.CT_from_t(aSA_2m, pwp_altr['temp'][2,tidx] - thresh, 2)
    arho_mld = gsw.rho(aSA_2m, aCT_mld, 2)

    # Find where the initial rho profile == the mld rho
    mld = pwp_orig['z_vector'][np.argmin(np.abs(ini_rho-(rho_mld)))]
    
    bl_mld =np.append(bl_mld,mld)

    amld = pwp_altr['z_vector'][np.argmin(np.abs(aini_rho-(arho_mld)))]
    
    nobl_mld=np.append(nobl_mld,amld)

    # isothermal layer depth defined by Rudzin 2018
    ild = -pwp_orig['z_vector'][(np.abs(pwp_orig['temp'][:,tidx] - pwp_orig['temp'][0,tidx]) > thresh)][0] 
    
    bl_ild =np.append(bl_ild,ild)

    aild = -pwp_altr['z_vector'][(np.abs(pwp_altr['temp'][:,tidx] - pwp_altr['temp'][0,tidx]) > thresh)][0] 
    
    nobl_ild=np.append(nobl_ild,aild)

## Example post-process plot: temperature response and MLD/ILD, barrier layer vs. no barrier layer

In [ ]:
fig,ax=plt.subplots(2,2,figsize=(15,8),sharex=True,sharey=True,constrained_layout=True)

vmin=26
vmax=31
tlevels = np.arange(26,31.5,0.5)

#### Top row: temperature cross-section with ILD/MLD overlay (barrier layer vs. no barrier layer)
c=ax[0,0].pcolormesh(pwp_orig.tgrid[0:60],-pwp_orig.zgrid[0:60],pwp_orig.temp[0:60],cmap=cmo.thermal,vmin=vmin,vmax=vmax)
c=ax[0,0].contourf(pwp_orig.tgrid[0:60],-pwp_orig.zgrid[0:60],pwp_orig.temp[0:60],cmap=cmo.thermal,vmin=vmin,vmax=vmax,levels=tlevels)
ax[0,0].set_xlim([mdates.datestr2num(tstart),mdates.datestr2num(plot_tend)])
ax[0,0].xaxis.set_major_formatter(myFmtlong)
ax[0,0].set_ylim(-60,0)
yticks=np.arange(-60,10,10)
ax[0,0].set_yticks(yticks)
ax[0,0].set_yticklabels(yticks,fontweight='bold')

#### plot 26 deg isotherm
CS=ax[0,0].contour(pwp_orig.tgrid[0:60],-pwp_orig.zgrid[0:60],pwp_orig.temp[0:60],[26],colors='white',linewidths=3)

ax[0,0].plot(pwp_orig['tgrid'][0,:],bl_ild,c='blue',label = 'ILD - BL',linewidth=3)
ax[0,0].text(pwp_orig['tgrid'][0,30]+.5,bl_ild[0],'ILD',{'color': 'black', 'fontsize': 15, 'ha': 'center', 'va': 'center',
          'bbox': dict(boxstyle="round", fc="white", ec="black", pad=0.2)})

ax[0,0].plot(pwp_orig['tgrid'][0,:],-bl_mld,c='blue', label = 'MLD - BL',linewidth=3)
ax[0,0].text(pwp_orig['tgrid'][0,30],-bl_mld[0],'MLD',{'color': 'black', 'fontsize': 15, 'ha': 'left', 'va': 'center',
          'bbox': dict(boxstyle="round", fc="white", ec="black", pad=0.2)})

ax[0,0].axvline(pwp_orig['tgrid'][0,ep_idx],ls='--',lw=3,color='gray',zorder=1)
ax[0,0].axvline(pwp_orig['tgrid'][0,lf_idx],lw=3,color='gray',zorder=1)

ax[0,0].set_ylabel('Depth [m]',fontweight='bold')
ax[0,0].text(pwp_orig.tgrid[0,4],-58,'a)',color='white',fontsize=17)

c1=ax[0,1].pcolormesh(pwp_altr.tgrid[0:60],-pwp_altr.zgrid[0:60],pwp_altr.temp[0:60],cmap=cmo.thermal,vmin=vmin,vmax=vmax)
c1=ax[0,1].contourf(pwp_altr.tgrid[0:60],-pwp_altr.zgrid[0:60],pwp_altr.temp[0:60],cmap=cmo.thermal,vmin=vmin,vmax=vmax,levels=tlevels)
cbar=plt.colorbar(c1,ax=ax[0,1])
cbar.ax.set_ylabel('Temperature [°C]',fontweight='bold')
for l in cbar.ax.yaxis.get_ticklabels():
    l.set_weight("bold")
    l.set_fontsize(17)
ax[0,1].set_xlim([mdates.datestr2num(tstart),mdates.datestr2num(plot_tend)])
ax[0,1].xaxis.set_major_formatter(myFmtlong)
ax[0,1].set_ylim(-60,0)

CS2=ax[0,1].contour(pwp_altr.tgrid[0:60],-pwp_altr.zgrid[0:60],pwp_altr.temp[0:60],[26],colors='white',linewidths=3)

ax[0,1].plot(pwp_orig['tgrid'][0,:],nobl_ild,c='blue',label='ILD - No BL',linewidth=3)
ax[0,1].text(pwp_orig['tgrid'][0,30]+.5,nobl_ild[0],'ILD',{'color': 'black', 'fontsize': 15, 'ha': 'center', 'va': 'center',
          'bbox': dict(boxstyle="round", fc="white", ec="black", pad=0.2)})

ax[0,1].plot(pwp_orig['tgrid'][0,:],-nobl_mld,c='blue', label='MLD - No BL',linewidth=3)
ax[0,1].text(pwp_orig['tgrid'][0,30],-nobl_mld[0],'MLD',{'color': 'black', 'fontsize': 15, 'ha': 'left', 'va': 'center',
          'bbox': dict(boxstyle="round", fc="white", ec="black", pad=0.2)})

ax[0,1].axvline(pwp_orig['tgrid'][0,ep_idx],ls='--',lw=3,color='gray',zorder=1)
ax[0,1].axvline(pwp_orig['tgrid'][0,lf_idx],lw=3,color='gray',zorder=1)

ax[0,1].text(pwp_orig.tgrid[0,4],-58,'b)',color='white',fontsize=17)

## clabels + panel titles
if save_prof == 1629339767: #8/19
    ax[0,0].clabel(CS, inline=True, fontsize=15,manual=[(18867,-55)])
    ax[0,1].clabel(CS2, inline=True, fontsize=15,manual=[(18867,-55)])
    ax[0,0].set_title('Exp1A',fontweight='bold')
    ax[0,1].set_title('Exp1B',fontweight='bold')
elif save_prof == 1630114503: #8/28
    ax[0,0].clabel(CS, inline=True, fontsize=15,manual=[(18869,-55)])
    ax[0,1].clabel(CS2, inline=True, fontsize=15,manual=[(18869,-55)])
    ax[0,0].set_title('Exp2A',fontweight='bold')
    ax[0,1].set_title('Exp2B',fontweight='bold')

#### Bottom row: change in temperature from t=0, same ILD/MLD overlay
tlims = (-1,1)

cur_plot = ax[1,0].pcolormesh(pwp_orig['tgrid'], -pwp_orig['zgrid'], pwp_orig['temp']-repmat(pwp_orig['temp'][:,0], np.shape(pwp_orig['temp'])[1],1).T, cmap=cmo.balance, vmin=tlims[0], vmax=tlims[1])

ax[1,0].set_ylabel('Depth [m]',fontweight='bold')

ax[1,0].plot(pwp_orig['tgrid'][0,:],bl_ild,c='blue',label = 'ILD - BL',linewidth=3)
ax[1,0].text(pwp_orig['tgrid'][0,30]+.5,bl_ild[0],'ILD',{'color': 'black', 'fontsize': 15, 'ha': 'center', 'va': 'center',
          'bbox': dict(boxstyle="round", fc="white", ec="black", pad=0.2)})

ax[1,0].plot(pwp_orig['tgrid'][0,:],-bl_mld,c='blue', label = 'MLD - BL',linewidth=3)
ax[1,0].text(pwp_orig['tgrid'][0,30],-bl_mld[0],'MLD',{'color': 'black', 'fontsize': 15, 'ha': 'left', 'va': 'center',
          'bbox': dict(boxstyle="round", fc="white", ec="black", pad=0.2)})

ax[1,0].axvline(pwp_orig['tgrid'][0,ep_idx],ls='--',lw=3,color='black',zorder=1)
ax[1,0].axvline(pwp_orig['tgrid'][0,lf_idx],lw=3,color='black',zorder=1)

ax[1,0].text(pwp_orig.tgrid[0,4],-58,'c)',color='black',fontsize=17)

xticks = pd.date_range(tstart,plot_tend,freq=tfreq)
ax[1,0].set_xticks(mdates.date2num(xticks))
ax[1,0].set_xticklabels(mdates.date2num(xticks),fontweight='bold')
ax[1,0].xaxis.set_major_formatter(myFmtlong)

yticks=np.arange(-60,10,10)
ax[1,0].set_yticks(yticks)
ax[1,0].set_yticklabels(yticks,fontweight='bold')
ax[1,0].tick_params(axis='x', which='major', pad=12)

cur_plot = ax[1,1].pcolormesh(pwp_altr['tgrid'], -pwp_altr['zgrid'], pwp_altr['temp']-repmat(pwp_altr['temp'][:,0], np.shape(pwp_altr['temp'])[1],1).T, cmap=cmo.balance, vmin=tlims[0], vmax=tlims[1])
cbar=plt.colorbar(cur_plot,ax=ax[1,1])
cbar.ax.set_ylabel('∆T(z)',fontweight='bold')
for l in cbar.ax.yaxis.get_ticklabels():
    l.set_weight("bold")
    l.set_fontsize(17)

ax[1,1].plot(pwp_orig['tgrid'][0,:],nobl_ild,c='blue',label='ILD - No BL',linewidth=3)
ax[1,1].text(pwp_orig['tgrid'][0,30]+.5,nobl_ild[0],'ILD',{'color': 'black', 'fontsize': 15, 'ha': 'center', 'va': 'center',
          'bbox': dict(boxstyle="round", fc="white", ec="black", pad=0.2)})

ax[1,1].plot(pwp_orig['tgrid'][0,:],-nobl_mld,c='blue', label='MLD - No BL',linewidth=3)
ax[1,1].text(pwp_orig['tgrid'][0,30],-nobl_mld[0],'MLD',{'color': 'black', 'fontsize': 15, 'ha': 'left', 'va': 'center',
          'bbox': dict(boxstyle="round", fc="white", ec="black", pad=0.2)})

ax[1,1].axvline(pwp_orig['tgrid'][0,ep_idx],ls='--',lw=3,color='black',zorder=1)
ax[1,1].axvline(pwp_orig['tgrid'][0,lf_idx],lw=3,color='black',zorder=1)

ax[1,1].text(pwp_orig.tgrid[0,4],-58,'d)',color='black',fontsize=17)

ax[1,1].set_xticks(mdates.date2num(xticks))
ax[1,1].set_xticklabels(mdates.date2num(xticks),fontweight='bold')
ax[1,1].xaxis.set_major_formatter(myFmtlong)
ax[1,1].tick_params(axis='x',which='major', pad=12)

plt.savefig(f'{FIGURES_DIR}/T_ILD_MLD_countours_'+str(save_prof)+'.png',dpi=300,bbox_inches='tight')